# 03 — Edge Case Analysis

**Goal:** Investigate deep causal structures and anomalies in the surgical action triplets.

We focus on three high-value analyses:
1. **Isolated Nodes:** Missing instruments or targets per video.
2. **Phase Transitions:** How the knowledge graph structure changes across surgical phases.
3. **Verb Transitions (Self-loops):** Causal action chains on the same instrument-target pair in consecutive frames.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import networkx as nx

sys.path.insert(0, os.path.abspath('..'))
from src.triplet_parser import load_categories
from src.edge_cases import get_missing_nodes, build_phase_subgraphs, find_verb_transitions
from src.graph_builder import print_graph_stats

# Paths
PROJECT_ROOT = Path('..').resolve()
TRIPLET_DIR = PROJECT_ROOT / 'outputs' / 'parsed_triplets'
JSON_DIR = PROJECT_ROOT / 'data' / 'CholecT45' / 'triplet'

# Load full categories from a JSON dict
categories = load_categories(JSON_DIR / 'VID01.json')

# Read the combined CSV to easily subset
df_all = pd.read_csv(TRIPLET_DIR / 'all_triplets.csv')

# Focus on non-null triplets
df_valid = df_all[~df_all['is_null']].copy()

## 1. Isolated / Missing Nodes

Which dictionary instruments and targets are completely missing from each video?

In [2]:
print('=== Missing Nodes per Video ===\n')
for vid in ['VID01', 'VID05', 'VID40']:
    df_vid = df_valid[df_valid['video'] == vid]
    missing = get_missing_nodes(df_vid, categories)
    print(f"{vid}:")
    print(f"  Missing Instruments: {missing['missing_instruments']}")
    print(f"  Missing Targets: {missing['missing_targets']}\n")

=== Missing Nodes per Video ===

VID01:
  Missing Instruments: []
  Missing Targets: ['adhesion', 'cystic_plate', 'gut', 'null_target', 'omentum', 'peritoneum']

VID05:
  Missing Instruments: []
  Missing Targets: ['adhesion', 'blood_vessel', 'null_target']

VID40:
  Missing Instruments: ['bipolar']
  Missing Targets: ['abdominal_wall_cavity', 'adhesion', 'blood_vessel', 'cystic_pedicle', 'gut', 'null_target', 'omentum', 'peritoneum']



## 2. Phase Transition Graphs

Surgical workflows are split into phases (Preparation, Calot Triangle Dissection, etc.).
We build a subgraph for each phase to see how the edges completely change as the surgery progresses.

In [3]:
vid01_data = df_valid[df_valid['video'] == 'VID01']

print(f'Total frames in VID01 with valid triplets: {vid01_data["frame"].nunique()}')
print('\nPhase distribution in VID01:')
print(vid01_data['phase'].value_counts())

phase_graphs = build_phase_subgraphs(vid01_data)
print(f'\nBuilt {len(phase_graphs)} phase subgraphs.\n')

for phase_name, G_phase in phase_graphs.items():
    print(f'--- {phase_name} ---')
    print(f'  Nodes: {G_phase.number_of_nodes()}')
    print(f'  Edges: {G_phase.number_of_edges()}')
    print(f'  Total Triplet Instances: {G_phase.graph.get("total_triplet_rows")}')


Total frames in VID01 with valid triplets: 1420

Phase distribution in VID01:
phase
gallbladder-dissection        931
carlot-triangle-dissection    770
clipping-and-cutting          266
gallbladder-packaging         113
cleaning-and-coagulation      108
gallbladder-extraction         42
preparation                    17
Name: count, dtype: int64

Built 7 phase subgraphs.

--- carlot-triangle-dissection ---
  Nodes: 9
  Edges: 8
  Total Triplet Instances: 770
--- cleaning-and-coagulation ---
  Nodes: 5
  Edges: 4
  Total Triplet Instances: 108
--- clipping-and-cutting ---
  Nodes: 5
  Edges: 4
  Total Triplet Instances: 266
--- gallbladder-dissection ---
  Nodes: 10
  Edges: 9
  Total Triplet Instances: 931
--- gallbladder-extraction ---
  Nodes: 2
  Edges: 1
  Total Triplet Instances: 42
--- gallbladder-packaging ---
  Nodes: 4
  Edges: 4
  Total Triplet Instances: 113
--- preparation ---
  Nodes: 2
  Edges: 1
  Total Triplet Instances: 17


## 3. Verb Transitions (Causal Action Chains)

We scan identical `(instrument, target)` pairs occurring in consecutive frames, looking for instances where the **verb changes**.
For example: `grasp` → `retract` on the gallbladder.

In [4]:
transitions = find_verb_transitions(vid01_data, max_frame_gap=1)

print(f'Found {len(transitions)} consecutive verb transitions in VID01.')
if not transitions.empty:
    # Show the most common transitions
    pathways = transitions.groupby(['instrument', 'verb_A', 'verb_B', 'target']).size().reset_index(name='count')
    pathways = pathways.sort_values('count', ascending=False)
    print('\nTop Causal Verb Pathways (Instrument -> V1 -> V2 -> Target):')
    display(pathways.head(10))
    
    print('\nSample Transition Timeline:')
    display(transitions[['frame_A', 'verb_A', 'frame_B', 'verb_B', 'target', 'phase_A']].head(5))

Found 40 consecutive verb transitions in VID01.

Top Causal Verb Pathways (Instrument -> V1 -> V2 -> Target):


,instrument,verb_A,verb_B,target,count
1,grasper,retract,grasp,gallbladder,34
0,grasper,grasp,retract,gallbladder,6



Sample Transition Timeline:


,frame_A,verb_A,frame_B,verb_B,target,phase_A
0,536,grasp,537,retract,gallbladder,carlot-triangle-dissection
1,551,grasp,552,retract,gallbladder,carlot-triangle-dissection
2,553,retract,554,grasp,gallbladder,carlot-triangle-dissection
3,559,retract,560,grasp,gallbladder,carlot-triangle-dissection
4,560,retract,561,grasp,gallbladder,carlot-triangle-dissection


---
**Conclusion:** The surgery is highly dynamic. The causal graph topology completely shifts depending on the phase, and strong causal signals exist in consecutive verb transitions.